In [ ]:
import json
import sys
from pathlib import Path

from kaggriculture_bot.evaluate import evaluate

sys.path.insert(0, str(Path.cwd().parent / "src"))

In [5]:
def my_agent(obs):
    market = []
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0) # Revisa si hay semillas en el inventario primero
    if obs["step"] == 0 or wheat_seeds <= 0: # Si no hay semillas, las compra en el mercado
        market.append(["BUY_SEED", "WHEAT", 1])
    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    tile = obs["farms"][obs["player"]]["tiles"][y][x]
    #print(tile)
    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL","WHEAT",1])

    if tile is None and wheat_seeds > 0: # Comprueba que no hay nada en la celda y que hay semilla en inventario
        farmer = ["PLANT", "WHEAT"] # Se planta la semilla
    elif tile is not None and tile.get("kind") == "PLANT" and tile.get("watered_today") == False: # Comprobación de si se ha regado
        farmer = ["WATER"] # Se riega la semilla
    elif tile is not None and tile.get("kind") == "PLANT" and obs["day"] - tile["planted_day"] >= 2: # Comprueba si han pasado más de 2 días desde que se plantó para cosechar.
        farmer = ["HARVEST"] # Se cosecha la semilla
    else: 
        farmer = ["PASS"]
    return {"farmer": farmer, "hands": [], "market": market}

In [16]:
def my_agent_v1(obs):
    market = []
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0) # Revisa si hay semillas en el inventario primero
    if obs["step"] == 0 or wheat_seeds <= 0: # Si no hay semillas, las compra en el mercado
        market.append(["BUY_SEED", "WHEAT", 1])
    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    tile = obs["farms"][obs["player"]]["tiles"][y][x]
    #print(tile)
    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL","WHEAT",wheat_in_shed])

    if tile is None and wheat_seeds > 0: # Comprueba que no hay nada en la celda y que hay semilla en inventario
        farmer = ["PLANT", "WHEAT"] # Se planta la semilla
    elif tile is not None and tile.get("kind") == "PLANT" and tile.get("watered_today") == False: # Comprobación de si se ha regado
        farmer = ["WATER"] # Se riega la semilla
    elif tile is not None and tile.get("kind") == "PLANT" and obs["day"] - tile["planted_day"] >= 2: # Comprueba si han pasado más de 2 días desde que se plantó para cosechar.
        farmer = ["HARVEST"] # Se cosecha la semilla
    else: 
        farmer = ["PASS"]
    return {"farmer": farmer, "hands": [], "market": market}

In [23]:
print("sell_one vs starter:", evaluate(my_agent, "starter", n=20))
print("sell_all vs starter:", evaluate(my_agent_v1, "starter", n=20))

sell_one vs starter: {'win_rate': 1.0, 'avg_money': 3880.0}
sell_all vs starter: {'win_rate': 1.0, 'avg_money': 3894.0}
